# Практика · Inception, MobileNet, EfficientNet

> Лекція: [lecture.html](lecture.html) · Тест: [quiz.html](quiz.html) ·
> Домашнє завдання: [homework.html](homework.html)

> ⏱ Зошит навчає **вісім** маленьких мереж і заміряє час девʼяти справжніх
> архітектур. Заміряно на чотирьох ядрах без відеокарти: **112, 136 і 173 секунди**
> у трьох прогонах, тобто **дві-три хвилини** залежно від того, наскільки зайнята
> машина. Рахунок іде в один потік — так швидше й відтворюваніше, див. нижче.

> ⚠️ Колонка **часу** в усіх таблицях нижче — про твою машину, а не про всі.
> Параметри й множення в тебе збіжаться з лекцією до останньої цифри; мілісекунди
> — ні, і це нормально. Важливий не сам час, а **порядок** мереж за ним.

Лекція стверджувала кілька речей числами. Тут ти дістанеш кожне число сам.

1. **Замір 1 — три «дешевше».** Параметри, множення й час для девʼяти справжніх
   архітектур. Побачимо, що порядок за параметрами **не той самий**, що за часом.
2. **Замір 2 — роздільна згортка.** Виведення формули, таблиця економії
   й **власна реалізація** згортки по глибині проти `nn.Conv2d(groups=…)`.
3. **Замір 3 — модуль Inception.** Зберемо руками, порахуємо параметри
   **рукою** й звіримо з `sum(p.numel())`. З горлом 1×1 і без нього.
4. **Замір 4 — на нашій задачі.** Дві мережі однакової форми: зі звичайними
   згортками й роздільними. Чи виправдалась економія?
5. **Замір 5 — множник ширини.** Чотири значення й розкид між зернами.
6. **Замір 6 — множник роздільності.** Параметри не змінюються, множення ростуть
   квадратично, а час — ні.

**Мережа не потрібна.** Архітектури будуються через `weights=None` — це створює
форму без жодного байта із інтернету. Датасет ми малюємо самі, формулами.

In [ ]:
import time
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models

# зерна фіксуємо на самому початку: без них числа нижче не збіжаться з лекцією
torch.manual_seed(0)

# Один потік, а не чотири. Причин дві, і обидві не залежать від навантаження:
# 1) на моделях у тисячі ваг багатопотоковість не дає виграшу — тензори замалі,
#    а координація потоків коштує;
# 2) під кількома потоками додавання float іде в іншому порядку, і числа
#    стрибають від прогону до прогону. Нам потрібна відтворюваність.
torch.set_num_threads(1)

notebook_started = time.perf_counter()

print("torch       :", torch.__version__)
print("numpy       :", np.__version__)
print("потоків CPU :", torch.get_num_threads())

## 1 · Три «дешевше»: параметри, множення, час

Почнемо з головного заміру теми. Нам потрібні три різні величини для однієї й тієї
самої мережі, і рахуються вони зовсім по-різному.

- **Параметри** — просто сума `numel()` по всіх тензорах ваг.
- **Множення** — рахуємо **самі**, за формулою по згорткових і лінійних шарах.
  Це чесніше за сторонню бібліотеку: ти бачиш, звідки взялося число, і можеш
  перевірити його на одному шарі рукою.
- **Час** — заміряємо прямий прохід і беремо **найкращий** із багатьох прогонів.
  Мінімум, а не середнє: середнє на завантаженій машині міряє чужі процеси, а не
  нашу мережу.

Спершу три функції-помічники.

In [ ]:
def count_parameters(model):
    """Скільки чисел треба зберігати — і як вони поділені між типами шарів."""
    total = sum(p.numel() for p in model.parameters())
    in_conv = sum(p.numel() for m in model.modules()
                  if isinstance(m, nn.Conv2d) for p in m.parameters())
    in_linear = sum(p.numel() for m in model.modules()
                    if isinstance(m, nn.Linear) for p in m.parameters())
    n_conv = sum(1 for m in model.modules() if isinstance(m, nn.Conv2d))
    return total, in_conv, in_linear, n_conv


def count_multiplications(model, image_size, in_channels=3):
    """Скільки множень мережа робить на одному зображенні.

    Формулу для згортки ми знаємо з теми 07:
        cout × H_out × W_out × (cin / groups) × k × k
    Розмір вихідної карти наперед невідомий, тому ми не вгадуємо його, а
    підвішуємо на кожен шар «гачок» (hook) і читаємо форму вже після проходу.
    """
    counted = {"conv": 0, "linear": 0}
    handles = []

    def on_conv(module, inputs, output):
        out_channels, height, width = output.shape[1], output.shape[2], output.shape[3]
        k_h, k_w = module.kernel_size
        counted["conv"] += (out_channels * height * width
                            * (module.in_channels // module.groups) * k_h * k_w)

    def on_linear(module, inputs, output):
        counted["linear"] += module.in_features * module.out_features

    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            handles.append(module.register_forward_hook(on_conv))
        elif isinstance(module, nn.Linear):
            handles.append(module.register_forward_hook(on_linear))

    model.eval()
    with torch.no_grad():
        model(torch.zeros(1, in_channels, image_size, image_size))
    for handle in handles:
        handle.remove()
    return counted["conv"], counted["linear"]


def measure_forward_ms(model, sample, repeats=15, warmup=3):
    """Час прямого проходу в мілісекундах — НАЙКРАЩИЙ із repeats прогонів.

    Мінімум, а не медіана: сплеск чужого навантаження може лише сповільнити
    наш прогін, тому найшвидший результат найближчий до правди про мережу.
    """
    model.eval()
    with torch.no_grad():
        for _ in range(warmup):
            model(sample)
        times = []
        for _ in range(repeats):
            started = time.perf_counter()
            model(sample)
            times.append((time.perf_counter() - started) * 1000)
    return min(times)


print("три помічники готові: параметри, множення, час")

Тепер збираємо девʼять справжніх архітектур. Ключове слово — `weights=None`:
`torchvision` будує **форму** мережі й заповнює її випадковими числами, нічого не
завантажуючи. Для нашого заміру цього досить: параметри, множення й час залежать
від форми, а не від того, чи мережа щось уміє.

Inception-v3 працює на вході 299×299 — це її рідний розмір, і міряти її на 224
було б нечесно.

In [ ]:
ARCHITECTURES = [
    ("AlexNet",            lambda: models.alexnet(weights=None),                     224),
    ("VGG16",              lambda: models.vgg16(weights=None),                       224),
    ("GoogLeNet",          lambda: models.googlenet(weights=None, init_weights=False), 224),
    ("ResNet18",           lambda: models.resnet18(weights=None),                    224),
    ("ResNet50",           lambda: models.resnet50(weights=None),                    224),
    ("Inception-v3",       lambda: models.inception_v3(weights=None, init_weights=False), 299),
    ("EfficientNet-B0",    lambda: models.efficientnet_b0(weights=None),             224),
    ("MobileNet-v2",       lambda: models.mobilenet_v2(weights=None),                224),
    ("MobileNet-v3-small", lambda: models.mobilenet_v3_small(weights=None),          224),
]

measured = []
print("%-20s %13s %8s %7s %6s %11s %9s %9s"
      % ("архітектура", "параметрів", "згортк.", "лінійн.", "conv", "множень,млн", "час,мс", "ваги,МБ"))
print("-" * 92)

for name, build, image_size in ARCHITECTURES:
    torch.manual_seed(0)
    model = build()
    total, in_conv, in_linear, n_conv = count_parameters(model)
    conv_mults, linear_mults = count_multiplications(model, image_size)
    all_mults = conv_mults + linear_mults
    sample = torch.randn(1, 3, image_size, image_size)
    milliseconds = measure_forward_ms(model, sample)
    megabytes = total * 4 / 1e6          # float32 — чотири байти на число

    measured.append(dict(name=name, size=image_size, params=total,
                         conv_share=in_conv / total, linear_share=in_linear / total,
                         n_conv=n_conv, mults=all_mults, ms=milliseconds, mb=megabytes))
    print("%-20s %13s %7.1f%% %6.1f%% %6d %11.1f %9.1f %9.1f"
          % (name, format(total, ",").replace(",", " "),
             100 * in_conv / total, 100 * in_linear / total, n_conv,
             all_mults / 1e6, milliseconds, megabytes))

Дві колонки посередині — це сюжет попередніх тем блоку, і він видно з першого
погляду: у **VGG16** 89.4 % ваг сидить у повнозвʼязних шарах, у **ResNet18** —
4.4 %. Ваги переїхали з голови в тіло.

А тепер головне. Впорядкуємо ті самі девʼять мереж **тричі** — за кожною з трьох
величин — і подивимось, чи збігаються порядки.

In [ ]:
def places(rows, key):
    """Місце кожної мережі в колонці: 1 — найдорожча."""
    order = sorted(range(len(rows)), key=lambda i: -rows[i][key])
    place = [0] * len(rows)
    for position, index in enumerate(order):
        place[index] = position + 1
    return place


by_params = places(measured, "params")
by_mults  = places(measured, "mults")
by_time   = places(measured, "ms")

print("%-20s %11s %11s %8s   %s" % ("архітектура", "за парам.", "за множ.", "за часом", "переїзд"))
print("-" * 70)
for i, row in enumerate(measured):
    travel = max(abs(by_params[i] - by_mults[i]),
                 abs(by_params[i] - by_time[i]),
                 abs(by_mults[i] - by_time[i]))
    mark = "  ←" if travel >= 3 else ""
    print("%-20s %11d %11d %8d   %d місць%s"
          % (row["name"], by_params[i], by_mults[i], by_time[i], travel, mark))

same = (by_params == by_mults == by_time)
print()
print("усі три порядки однакові?", same)
assert not same, "порядки збіглися — головне твердження теми не відтворилось!"
print("✅ порядки РІЗНІ — саме це й стверджувала лекція")

Найбільший переїзд робить **AlexNet**. Подивимось, звідки він береться, на двох
шарах цієї самої мережі — руками, без бібліотек.

In [ ]:
# Повнозвʼязний шар AlexNet: 9216 входів → 4096 виходів.
dense_weights = 9216 * 4096
dense_mults = 9216 * 4096          # одна вага — рівно одне множення

# Згортковий шар типової глибокої мережі: 256 каналів → 256, ядро 3×3,
# карта 28×28 (стільки лишається від 224×224 після трьох зменшень удвічі).
conv_weights = 256 * 256 * 3 * 3
conv_mults = conv_weights * 28 * 28   # те саме ядро прикладається до кожного положення

print("повнозвʼязний 9216→4096 : %11s ваг, %13s множень  (%.1f множення на вагу)"
      % (format(dense_weights, ",").replace(",", " "),
         format(dense_mults, ",").replace(",", " "), dense_mults / dense_weights))
print("згортковий 256→256 3×3   : %11s ваг, %13s множень  (%.0f множень на вагу)"
      % (format(conv_weights, ",").replace(",", " "),
         format(conv_mults, ",").replace(",", " "), conv_mults / conv_weights))
print()
print("Ось і вся розгадка: у повнозвʼязному шарі вага працює ОДИН раз,")
print("у згортковому — %d разів. Тому мережа з важкою головою важить багато," % (28 * 28))
print("а рахує мало, і навпаки.")

І остання перевірка цього розділу: поділимо множення на час. Вийде «скільки
мільйонів множень залізо встигає за мілісекунду» — тобто наскільки повно
використано процесор.

In [ ]:
print("%-20s %12s %9s %14s" % ("архітектура", "множень,млн", "час,мс", "млн множ./мс"))
print("-" * 60)
throughput = []
for row in sorted(measured, key=lambda r: -r["mults"] / r["ms"]):
    speed = row["mults"] / 1e6 / row["ms"]
    throughput.append(speed)
    print("%-20s %12.1f %9.1f %14.1f" % (row["name"], row["mults"] / 1e6, row["ms"], speed))

print()
print("розкид: від %.1f до %.1f, тобто в %.1f раза"
      % (min(throughput), max(throughput), max(throughput) / min(throughput)))
print()
print("Найповільніші за цією міркою — мережі на роздільних згортках.")
print("Чому саме вони, розберемо в розділі 2.")

### Дрібниця, яку легко процитувати неправильно

`torchvision` за замовчуванням будує GoogLeNet і Inception-v3 **разом із
допоміжними класифікаторами** — додатковими головами, приліпленими до середини
мережі, щоб градієнт доходив до перших шарів. Працюють вони лише під час
навчання. Порахуємо обидва числа.

In [ ]:
for label, with_aux, without_aux in [
    ("GoogLeNet",
     models.googlenet(weights=None, init_weights=False),
     models.googlenet(weights=None, init_weights=False, aux_logits=False)),
    ("Inception-v3",
     models.inception_v3(weights=None, init_weights=False),
     models.inception_v3(weights=None, init_weights=False, aux_logits=False)),
]:
    a = sum(p.numel() for p in with_aux.parameters())
    b = sum(p.numel() for p in without_aux.parameters())
    print("%-14s із допоміжними головами: %12s   без них: %12s   різниця: %s"
          % (label,
             format(a, ",").replace(",", " "),
             format(b, ",").replace(",", " "),
             format(a - b, ",").replace(",", " ")))
print()
print("У GoogLeNet майже половина «офіційних» параметрів на пристрій ніколи не потрапляє.")

## 2 · Роздільна згортка по глибині

Звичайна згортка робить дві справи одночасно: дивиться на **сусідів у просторі**
(вікно `k×k`) і змішує **канали** між собою. Роздільна згортка ділить це на два
кроки: спершу тільки простір, окремо в кожному каналі; потім тільки канали,
без огляду на сусідів.

Порахуємо параметри обох варіантів за формулами — і одразу звіримо з тим, що дає
`torch`. Якщо наша формула розійдеться з бібліотекою, ми про це дізнаємось тут,
а не через три розділи.

In [ ]:
def regular_conv_parameters(c_in, c_out, k):
    """Звичайна згортка: c_in × c_out ядер розміру k×k, плюс по зсуву на канал."""
    return c_in * c_out * k * k + c_out


def separable_conv_parameters(c_in, c_out, k):
    """Роздільна: (по глибині: одне ядро k×k на канал) + (точкова 1×1)."""
    depthwise = c_in * k * k + c_in
    pointwise = c_in * c_out + c_out
    return depthwise + pointwise


print("%-12s %12s %12s %9s %s" % ("вхід→вихід", "звичайна", "роздільна", "різниця", "звірка з torch"))
print("-" * 66)
for c_in, c_out in [(32, 64), (64, 128), (128, 256), (256, 512)]:
    ours_regular = regular_conv_parameters(c_in, c_out, 3)
    ours_separable = separable_conv_parameters(c_in, c_out, 3)

    torch_regular = sum(p.numel() for p in nn.Conv2d(c_in, c_out, 3, padding=1).parameters())
    torch_separable = sum(p.numel() for p in nn.Sequential(
        nn.Conv2d(c_in, c_in, 3, padding=1, groups=c_in),   # groups=c_in — кожен канал сам по собі
        nn.Conv2d(c_in, c_out, 1)).parameters())

    assert ours_regular == torch_regular and ours_separable == torch_separable
    print("%-12s %12s %12s %8.2f× ✅"
          % ("%d→%d" % (c_in, c_out),
             format(ours_regular, ",").replace(",", " "),
             format(ours_separable, ",").replace(",", " "),
             ours_regular / ours_separable))

Числа наближаються до девʼяти й не переходять межу. Чому саме девʼять — видно
з формул, якщо на секунду відкинути зсуви:

```
    c_in·c_out·k²                    1
  ─────────────────────  =  ──────────────────
  c_in·k² + c_in·c_out       1/k²  +  1/c_out
```

Кількість **вхідних** каналів скоротилась і на відношення не впливає взагалі.
Лишились розмір ядра й кількість **вихідних** каналів. Перевіримо це числом.

In [ ]:
def ratio_limit(c_out, k):
    """Межа економії без урахування зсувів."""
    return 1 / (1 / (k * k) + 1 / c_out)


print("вхідні канали НЕ впливають (k=3, c_out=128):")
for c_in in [16, 64, 256, 1024]:
    r = regular_conv_parameters(c_in, 128, 3) / separable_conv_parameters(c_in, 128, 3)
    print("   c_in = %5d  →  %.2f×" % (c_in, r))

print()
print("вихідні канали впливають (k=3, c_in=64):")
for c_out in [8, 32, 128, 512, 4096]:
    r = regular_conv_parameters(64, c_out, 3) / separable_conv_parameters(64, c_out, 3)
    print("   c_out = %5d  →  %.2f×   (межа за формулою %.2f×, стеля k² = 9)"
          % (c_out, r, ratio_limit(c_out, 3)))

### Власна реалізація: перевіримо, що всередині бібліотеки немає магії

Напишемо згортку по глибині й точкову згортку **самі**, на чистому `numpy`,
циклами — так, щоб було видно кожен доданок. Потім порівняємо з
`nn.Conv2d(groups=…)` через `np.allclose`.

In [ ]:
def depthwise_by_hand(image, kernels):
    """Просторова згортка окремо в кожному каналі.

    image   — (C, H, W), kernels — (C, k, k): своє ядро на кожен канал.
    Канали між собою не змішуються — у цьому вся суть кроку «де».
    """
    channels, height, width = image.shape
    k = kernels.shape[1]
    pad = k // 2
    padded = np.pad(image, ((0, 0), (pad, pad), (pad, pad)))
    result = np.zeros((channels, height, width))

    for channel in range(channels):
        for row in range(height):
            for col in range(width):
                window = padded[channel, row:row + k, col:col + k]
                result[channel, row, col] = np.sum(window * kernels[channel])
    return result


def pointwise_by_hand(feature_maps, mixing):
    """Змішування каналів у кожному пікселі окремо.

    feature_maps — (C_in, H, W), mixing — (C_out, C_in).
    Просторових сусідів цей крок не бачить зовсім — це крок «що».
    """
    out_channels = mixing.shape[0]
    in_channels, height, width = feature_maps.shape
    result = np.zeros((out_channels, height, width))

    for out_channel in range(out_channels):
        for in_channel in range(in_channels):
            result[out_channel] += mixing[out_channel, in_channel] * feature_maps[in_channel]
    return result


torch.manual_seed(7)
C_IN, C_OUT, K, SIDE = 4, 6, 3, 9

# бібліотечний варіант: два шари поспіль, без зсувів — щоб порівнювати чисту згортку
library_depthwise = nn.Conv2d(C_IN, C_IN, K, padding=K // 2, groups=C_IN, bias=False)
library_pointwise = nn.Conv2d(C_IN, C_OUT, 1, bias=False)

sample_image = torch.randn(1, C_IN, SIDE, SIDE)
with torch.no_grad():
    library_result = library_pointwise(library_depthwise(sample_image))[0].numpy()

# ті самі ваги, витягнуті у форму, зручну для наших циклів
# у згортці з groups=C_IN кожна група має рівно один вхідний канал, тому [:, 0]
kernels = library_depthwise.weight.detach().numpy()[:, 0]
mixing = library_pointwise.weight.detach().numpy()[:, :, 0, 0]

our_result = pointwise_by_hand(depthwise_by_hand(sample_image[0].numpy(), kernels), mixing)

print("наша форма       :", our_result.shape)
print("бібліотечна форма:", library_result.shape)
print("максимальна розбіжність: %.2e" % np.abs(our_result - library_result).max())
assert np.allclose(our_result, library_result, atol=1e-5), "розрахунок розійшовся!"
print("✅ збігається")

### А тепер час: економія параметрів ≠ економія часу

Параметрів у 8.24 раза менше. Множень — приблизно так само. Заміряймо час на
реальному вході: 64 канали, карта 56×56.

In [ ]:
input_map = torch.randn(1, 64, 56, 56)
regular_layer = nn.Conv2d(64, 128, 3, padding=1)
separable_layer = nn.Sequential(
    nn.Conv2d(64, 64, 3, padding=1, groups=64),
    nn.Conv2d(64, 128, 1))

regular_ms = measure_forward_ms(regular_layer, input_map, repeats=30, warmup=5)
separable_ms = measure_forward_ms(separable_layer, input_map, repeats=30, warmup=5)

params_ratio = (regular_conv_parameters(64, 128, 3)
                / separable_conv_parameters(64, 128, 3))

print("вхід 64×56×56, один потік CPU")
print("   звичайна 3×3 : %8s параметрів, %6.2f мс"
      % (format(regular_conv_parameters(64, 128, 3), ",").replace(",", " "), regular_ms))
print("   роздільна    : %8s параметрів, %6.2f мс"
      % (format(separable_conv_parameters(64, 128, 3), ",").replace(",", " "), separable_ms))
print()
print("   параметрів менше в %.2f раза" % params_ratio)
print("   часу менше в      %.2f раза" % (regular_ms / separable_ms))
print()
print("Виграш за часом ЗАВЖДИ менший за виграш за параметрами. Причина в тому,")
print("що процесор витрачає час не на множення, а на очікування чисел із памʼяті:")
print("згортка по глибині читає ту саму карту, а роботи на прочитаний піксель")
# на кожен прочитаний піксель звичайна згортка робить c_out×k² множень,
# а згортка по глибині — лише k²; відношення дорівнює просто c_out
print("робить у %d разів менше." % 128)

## 3 · Модуль Inception і пляшкове горло 1×1

Модуль Inception пропускає вхід через чотири гілки одразу — 1×1, 3×3, 5×5
і пулінг — а потім склеює їхні виходи **по каналах** через `torch.cat`.

Зберемо його руками, у двох варіантах: із горлом 1×1 перед дорогими ядрами
й без нього.

In [ ]:
class InceptionModule(nn.Module):
    """Модуль Inception 3a з GoogLeNet.

    bottleneck=True  — перед ядрами 3×3 і 5×5 стоїть згортка 1×1, яка стискає
                       кількість каналів (те саме «пляшкове горло»);
    bottleneck=False — ті самі ядра прикладаються до всіх вхідних каналів одразу.
    """

    def __init__(self, c_in, out_1x1, reduce_3x3, out_3x3,
                 reduce_5x5, out_5x5, out_pool, bottleneck=True):
        super().__init__()
        self.branch_1x1 = nn.Conv2d(c_in, out_1x1, 1)

        if bottleneck:
            self.branch_3x3 = nn.Sequential(
                nn.Conv2d(c_in, reduce_3x3, 1), nn.ReLU(),
                nn.Conv2d(reduce_3x3, out_3x3, 3, padding=1))
            self.branch_5x5 = nn.Sequential(
                nn.Conv2d(c_in, reduce_5x5, 1), nn.ReLU(),
                nn.Conv2d(reduce_5x5, out_5x5, 5, padding=2))
        else:
            self.branch_3x3 = nn.Conv2d(c_in, out_3x3, 3, padding=1)
            self.branch_5x5 = nn.Conv2d(c_in, out_5x5, 5, padding=2)

        # пулінг ваг не має, тому гілка з ним однакова в обох варіантах
        self.branch_pool = nn.Sequential(
            nn.MaxPool2d(3, stride=1, padding=1),
            nn.Conv2d(c_in, out_pool, 1))

    def forward(self, x):
        # dim=1 — це вимір каналів: стоси лягають один на одного, нічого не додається
        return torch.cat([self.branch_1x1(x), self.branch_3x3(x),
                          self.branch_5x5(x), self.branch_pool(x)], dim=1)


CONFIG = dict(c_in=192, out_1x1=64, reduce_3x3=96, out_3x3=128,
              reduce_5x5=16, out_5x5=32, out_pool=32)

with_bottleneck = InceptionModule(**CONFIG, bottleneck=True)
without_bottleneck = InceptionModule(**CONFIG, bottleneck=False)

probe = torch.randn(1, 192, 28, 28)
print("вхід :", tuple(probe.shape))
print("вихід:", tuple(with_bottleneck(probe).shape),
      " ← 64 + 128 + 32 + 32 = %d каналів, карта та сама 28×28" % (64 + 128 + 32 + 32))

Тепер порахуємо параметри модуля **рукою**, гілка за гілкою, і звіримо з тим,
що скаже `sum(p.numel())`. Формула та сама: `c_in × c_out × k² + c_out`.

In [ ]:
c_in = 192

# ── з горлом 1×1 ──
branch_1x1_hand = 192 * 64 + 64
branch_3x3_hand = (192 * 96 + 96) + (96 * 128 * 9 + 128)      # стискаємо 192→96, потім 3×3
branch_5x5_hand = (192 * 16 + 16) + (16 * 32 * 25 + 32)       # стискаємо 192→16, потім 5×5
branch_pool_hand = 192 * 32 + 32
hand_with = branch_1x1_hand + branch_3x3_hand + branch_5x5_hand + branch_pool_hand

# ── без горла ──
branch_3x3_direct = 192 * 128 * 9 + 128
branch_5x5_direct = 192 * 32 * 25 + 32
hand_without = branch_1x1_hand + branch_3x3_direct + branch_5x5_direct + branch_pool_hand

torch_with = sum(p.numel() for p in with_bottleneck.parameters())
torch_without = sum(p.numel() for p in without_bottleneck.parameters())

print("%-14s %12s %12s %9s" % ("гілка", "без горла", "з горлом", "різниця"))
print("-" * 52)
print("%-14s %12s %12s %9s" % ("1×1 → 64",
      format(branch_1x1_hand, ",").replace(",", " "),
      format(branch_1x1_hand, ",").replace(",", " "), "—"))
print("%-14s %12s %12s %8.1f×" % ("3×3 → 128",
      format(branch_3x3_direct, ",").replace(",", " "),
      format(branch_3x3_hand, ",").replace(",", " "),
      branch_3x3_direct / branch_3x3_hand))
print("%-14s %12s %12s %8.1f×" % ("5×5 → 32",
      format(branch_5x5_direct, ",").replace(",", " "),
      format(branch_5x5_hand, ",").replace(",", " "),
      branch_5x5_direct / branch_5x5_hand))
print("%-14s %12s %12s %9s" % ("pool → 32",
      format(branch_pool_hand, ",").replace(",", " "),
      format(branch_pool_hand, ",").replace(",", " "), "—"))
print("-" * 52)
print("%-14s %12s %12s %8.2f×" % ("модуль",
      format(hand_without, ",").replace(",", " "),
      format(hand_with, ",").replace(",", " "),
      hand_without / hand_with))
print()
print("ручний рахунок з горлом : %s   torch: %s"
      % (format(hand_with, ",").replace(",", " "),
         format(torch_with, ",").replace(",", " ")))
print("ручний рахунок без горла: %s   torch: %s"
      % (format(hand_without, ",").replace(",", " "),
         format(torch_without, ",").replace(",", " ")))
assert hand_with == torch_with and hand_without == torch_without, "ручний рахунок розійшовся!"
print("✅ збігається до останнього числа")

Горло дає 2.4 раза на цьому модулі. Але чи завжди воно вигідне? Витрата на нього
не залежить від того, скільки каналів ми стискаємо, а виграш — залежить.
Прокрутимо кількість вхідних каналів.

In [ ]:
def inception_parameters(c_in, bottleneck):
    """Параметри модуля Inception 3a при довільній кількості вхідних каналів."""
    module = InceptionModule(c_in=c_in, out_1x1=64, reduce_3x3=96, out_3x3=128,
                             reduce_5x5=16, out_5x5=32, out_pool=32,
                             bottleneck=bottleneck)
    return sum(p.numel() for p in module.parameters())


print("%8s %12s %12s %10s   %s" % ("каналів", "без горла", "з горлом", "економія", "висновок"))
print("-" * 66)
for channels in [32, 48, 64, 80, 128, 192, 320, 512]:
    without = inception_parameters(channels, False)
    with_it = inception_parameters(channels, True)
    ratio = without / with_it
    verdict = "горло окупається" if ratio > 1.02 else (
              "нічого не дає" if ratio > 0.98 else "горло ЛИШЕ ШКОДИТЬ")
    print("%8d %12s %12s %9.2f×   %s"
          % (channels,
             format(without, ",").replace(",", " "),
             format(with_it, ",").replace(",", " "), ratio, verdict))

print()
print("На вузькому вході стискати нема чого, а зайвий шар 1×1 усе одно треба")
print("оплатити. Тому горло стоїть углиб мережі, а не на її початку.")

## 4 · Наша задача: фігури 28×28

Далі ми переходимо від розбору чужих архітектур до власних замірів. Задача — та
сама, що в усьому блоці: шість фігур на 28×28, намальованих формулами. Зсув
центра ±4 пікселі, радіус 5-8, гаусів шум σ = 0.35.

Шум тут не для краси. На чистих фігурах будь-яка з наших мереж дає одиницю,
і жодної різниці між варіантами не видно.

In [ ]:
SHAPE_NAMES = ["коло", "квадрат", "ромб", "кільце", "хрест", "трикутник"]
NOISE, JITTER = 0.35, 4


def draw_shape(kind, rng, size=28):
    """Малює одну фігуру як масив size×size зі значеннями 0..1.

    Розміри масштабуються разом зі стороною — щоб фігура на вході 20×20 була
    тією самою фігурою, тільки в меншій роздільності, а не іншою задачею.
    """
    image = np.zeros((size, size), dtype=np.float32)
    scale = size / 28.0
    center_y = size / 2 + rng.integers(-JITTER, JITTER + 1) * scale
    center_x = size / 2 + rng.integers(-JITTER, JITTER + 1) * scale
    radius = rng.integers(5, 9) * scale

    # відстані кожного пікселя від центра — з них складаються всі шість фігур
    yy, xx = np.mgrid[0:size, 0:size]
    dy, dx = yy - center_y, xx - center_x

    if kind == 0:                                    # коло
        image[dy * dy + dx * dx <= radius * radius] = 1.0
    elif kind == 1:                                  # квадрат
        image[(np.abs(dy) <= radius * 0.85) & (np.abs(dx) <= radius * 0.85)] = 1.0
    elif kind == 2:                                  # ромб
        image[np.abs(dy) + np.abs(dx) <= radius] = 1.0
    elif kind == 3:                                  # кільце
        distance = dy * dy + dx * dx
        image[(distance <= radius * radius)
              & (distance >= (radius - 3 * scale) ** 2)] = 1.0
    elif kind == 4:                                  # хрест
        image[(np.abs(dy) <= 2 * scale) & (np.abs(dx) <= radius)] = 1.0
        image[(np.abs(dx) <= 2 * scale) & (np.abs(dy) <= radius)] = 1.0
    else:                                            # трикутник
        image[(dy >= -radius * 0.8) & (dy <= radius * 0.8)
              & (np.abs(dx) <= (dy + radius * 0.8) * 0.6)] = 1.0

    image += rng.normal(0, NOISE, image.shape).astype(np.float32)
    return np.clip(image, 0, 1)


def make_dataset(count, size, seed):
    """Повертає (count, 1, size, size) і (count,). Класи йдуть порівну, по колу."""
    rng = np.random.default_rng(seed)
    images = np.zeros((count, 1, size, size), dtype=np.float32)
    labels = np.zeros(count, dtype=np.int64)
    for index in range(count):
        images[index, 0] = draw_shape(index % 6, rng, size)
        labels[index] = index % 6
    return torch.from_numpy(images), torch.from_numpy(labels)


generation_started = time.perf_counter()
DATA = {}
for side in (20, 28):
    DATA[side] = (make_dataset(1200, side, 42), make_dataset(1200, side, 7))
print("датасети згенеровано за %.1f с" % (time.perf_counter() - generation_started))
for side in (20, 28):
    (train_x, train_y), (valid_x, valid_y) = DATA[side]
    print("   %2d×%-2d : навчання %s, перевірка %s"
          % (side, side, tuple(train_x.shape), tuple(valid_x.shape)))

In [ ]:
import matplotlib.pyplot as plt

(train_x, train_y), _ = DATA[28]
figure, axes = plt.subplots(1, 6, figsize=(11, 2.1))
for slot in range(6):
    axes[slot].imshow(train_x[slot, 0], cmap="gray", vmin=0, vmax=1)
    axes[slot].set_title(SHAPE_NAMES[int(train_y[slot])], fontsize=9)
    axes[slot].axis("off")
plt.tight_layout()
plt.show()
print("шість класів, шум σ =", NOISE, "— фігуру видно, але не одразу")

Тепер сама мережа. Вона зібрана так, щоб той самий код давав і звичайний, і
роздільний варіант, і щоб кількість каналів множилась на один параметр.

Голова — `AdaptiveAvgPool2d(1)` і один лінійний шар. Глобальне усереднення
з [теми 08](../08-pooling/lecture.html) тут не прикраса: саме завдяки йому
мережа приймає вхід **будь-якого** розміру, і кількість параметрів від нього не
залежить. Це знадобиться в розділі 7.

In [ ]:
def regular_block(c_in, c_out):
    """Звичайний блок: одна згортка 3×3, що робить «де» і «що» одночасно."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False),
        nn.BatchNorm2d(c_out), nn.ReLU())


def separable_block(c_in, c_out):
    """Роздільний блок: спершу тільки «де», потім тільки «що»."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_in, 3, padding=1, groups=c_in, bias=False),   # по глибині
        nn.BatchNorm2d(c_in), nn.ReLU(),
        nn.Conv2d(c_in, c_out, 1, bias=False),                          # точкова
        nn.BatchNorm2d(c_out), nn.ReLU())


class ShapeNet(nn.Module):
    """Маленька CNN на шість фігур.

    width      — множник ширини: на нього множиться кількість каналів у кожному шарі;
    separable  — чи брати роздільні згортки замість звичайних.

    Перший блок лишається звичайним завжди: на вході в нього один канал, і ділити
    там нічого. MobileNet робить так само.
    """

    def __init__(self, width=1.0, separable=False, n_classes=6):
        super().__init__()
        c1, c2, c3 = [max(4, int(round(c * width))) for c in (16, 32, 64)]
        block = separable_block if separable else regular_block
        self.body = nn.Sequential(
            regular_block(1, c1), nn.MaxPool2d(2),
            block(c1, c2), nn.MaxPool2d(2),
            block(c2, c3),
            nn.AdaptiveAvgPool2d(1))          # карта будь-якого розміру → одне число на канал
        self.head = nn.Sequential(nn.Flatten(), nn.Linear(c3, n_classes))
        self.channels = (c1, c2, c3)

    def forward(self, x):
        return self.head(self.body(x))


def train(model, x, y, epochs=12, learning_rate=1e-3, batch=32, seed=0):
    """Навчає мережу й повертає її. Зерно керує і початковими вагами, і порядком батчів."""
    torch.manual_seed(seed)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_function = nn.CrossEntropyLoss()
    model.train()
    for _ in range(epochs):
        order = torch.randperm(len(x))
        for start in range(0, len(x), batch):
            chosen = order[start:start + batch]
            optimizer.zero_grad()
            loss_function(model(x[chosen]), y[chosen]).backward()
            optimizer.step()
    return model


@torch.no_grad()
def accuracy(model, x, y):
    """Частка правильних відповідей на перевірочній вибірці."""
    model.eval()
    return (model(x).argmax(1) == y).float().mean().item()


def run_experiment(label, width, separable, side, seed=0):
    """Один повний дослід: побудувати, порахувати, навчити, заміряти."""
    (train_x, train_y), (valid_x, valid_y) = DATA[side]
    torch.manual_seed(seed)
    model = ShapeNet(width=width, separable=separable)
    parameters = sum(p.numel() for p in model.parameters())
    conv_mults, linear_mults = count_multiplications(model, side, in_channels=1)

    started = time.perf_counter()
    train(model, train_x, train_y, seed=seed)
    training_seconds = time.perf_counter() - started

    result = dict(label=label, width=width, separable=separable, side=side, seed=seed,
                  channels=model.channels, params=parameters,
                  mults=conv_mults + linear_mults,
                  train_s=training_seconds,
                  ms=measure_forward_ms(model, valid_x[:64], repeats=40, warmup=5),
                  accuracy=accuracy(model, valid_x, valid_y))
    print("   %-22s канали %-14s %7s парам · %9s множень · навчання %5.1f с · "
          "прохід %5.2f мс · точність %.3f"
          % (label, str(model.channels),
             format(parameters, ",").replace(",", " "),
             format(result["mults"], ",").replace(",", " "),
             training_seconds, result["ms"], result["accuracy"]))
    return result


experiments = []
print("мережа й функції навчання готові")

## 5 · Замір: звичайні згортки проти роздільних

Дві мережі однакової форми, однакової ширини, на тих самих даних. Відрізняються
лише тим, які згортки стоять у другому й третьому блоках.

⏱ Ця клітинка навчає **дві** мережі — близько сорока секунд.

In [ ]:
print("навчаю дві мережі однакової форми:")
regular_run = run_experiment("звичайні згортки", width=1.0, separable=False, side=28)
separable_run = run_experiment("роздільні згортки", width=1.0, separable=True, side=28)
experiments += [regular_run, separable_run]

print()
print("%-20s %10s %12s %10s %10s" % ("", "параметрів", "множень", "прохід,мс", "точність"))
print("-" * 66)
for run in (regular_run, separable_run):
    print("%-20s %10s %12s %10.2f %10.3f"
          % (run["label"],
             format(run["params"], ",").replace(",", " "),
             format(run["mults"], ",").replace(",", " "),
             run["ms"], run["accuracy"]))
print("-" * 66)
print("%-20s %9.2f× %11.2f× %9.2f× %+10.3f"
      % ("різниця",
         regular_run["params"] / separable_run["params"],
         regular_run["mults"] / separable_run["mults"],
         regular_run["ms"] / separable_run["ms"],
         separable_run["accuracy"] - regular_run["accuracy"]))
print()
print("Ось відповідь на питання «чи виправдалась економія»:")
print("  за параметрами — так, і дуже;")
print("  за множеннями  — так;")
print("  за часом       — майже ні;")
print("  за точністю    — нічого не втрачено.")

## 6 · Замір: множник ширини

Множник ширини — це число, на яке множиться кількість каналів у кожному шарі.
Параметри згорткового шару — `c_in × c_out × k²`, і множник зменшує **обидва**
множники одразу, тож параметрів стає приблизно у `α²` разів менше.

⏱ Клітинка навчає **три** мережі — близько сорока секунд. Четверта, з
`α = 1.0`, уже навчена в попередньому розділі.

In [ ]:
print("навчаю три вужчі мережі:")
width_runs = [run_experiment("ширина %.2f" % w, width=w, separable=False, side=28)
              for w in (0.25, 0.50, 0.75)]
width_runs.append(regular_run)                # α = 1.0 вже є
experiments += width_runs[:3]

print()
print("%8s %14s %10s %12s %10s %10s"
      % ("множник", "канали", "параметрів", "множень", "прохід,мс", "точність"))
print("-" * 72)
for width, run in zip((0.25, 0.50, 0.75, 1.00), width_runs):
    print("%8.2f %14s %10s %12s %10.2f %10.3f"
          % (width, str(run["channels"]),
             format(run["params"], ",").replace(",", " "),
             format(run["mults"], ",").replace(",", " "),
             run["ms"], run["accuracy"]))

### Обережно: чи справжня ця різниця?

Точність при `α = 0.5` і при `α = 1.0` відрізняється приблизно на пункт. Перш ніж
робити з цього висновок, треба спитати, наскільки взагалі стійке саме число.
Навчимо ту саму мережу при `α = 0.5` ще двічі, змінивши **лише випадкове зерно**.

⏱ Ще **дві** мережі — близько двадцяти секунд.

In [ ]:
print("та сама мережа, інші зерна:")
seed_runs = [run_experiment("ширина 0.50, зерно %d" % s, width=0.5,
                            separable=False, side=28, seed=s)
             for s in (1, 2)]
experiments += seed_runs

same_width = [width_runs[1]["accuracy"]] + [r["accuracy"] for r in seed_runs]
spread = max(same_width) - min(same_width)
gap = abs(width_runs[3]["accuracy"] - width_runs[1]["accuracy"])

print()
print("точність при α = 0.5 на трьох зернах:", " ".join("%.3f" % a for a in same_width))
print("розкид між зернами            : %.3f" % spread)
print("різниця між α = 0.5 і α = 1.0 : %.3f" % gap)
print()
if gap < spread:
    print("Розкид БІЛЬШИЙ за різницю. Отже втрату від половинної ширини на цій")
    print("задачі НЕ ДОВЕДЕНО — вона тоне у випадковості навчання.")
else:
    print("Різниця більша за розкид — втрата від половинної ширини справжня.")

drop = width_runs[3]["accuracy"] - width_runs[0]["accuracy"]
print()
print("А от падіння при α = 0.25 дорівнює %.3f — це %.1f розкиду, і воно справжнє."
      % (drop, drop / spread))

## 7 · Замір: множник роздільності

Друга ручка MobileNet — менший вхід. Діє вона зовсім інакше за першу:

| | множник ширини | множник роздільності |
|---|---|---|
| параметри | падають | **не змінюються** |
| множення | падають | падають |

Спершу без жодного навчання: візьмемо одну мережу й подамо їй вісім різних
розмірів входу. Параметри мають лишитись незмінними — за це відповідає глобальне
усереднення.

In [ ]:
torch.manual_seed(0)
probe_net = ShapeNet(width=1.0, separable=False)
probe_params = sum(p.numel() for p in probe_net.parameters())

print("параметрів у мережі: %s — і це число нижче не зміниться жодного разу"
      % format(probe_params, ",").replace(",", " "))
print()
print("%8s %12s %10s %12s %10s"
      % ("вхід", "множень", "час,мс", "множень,×", "часу,×"))
print("-" * 58)

resolution_rows = []
for side in (16, 20, 24, 28, 32, 40, 48, 56):
    conv_mults, linear_mults = count_multiplications(probe_net, side, in_channels=1)
    milliseconds = measure_forward_ms(probe_net, torch.randn(1, 1, side, side),
                                      repeats=60, warmup=10)
    same = sum(p.numel() for p in probe_net.parameters())
    assert same == probe_params, "параметри змінилися від розміру входу!"
    resolution_rows.append((side, conv_mults + linear_mults, milliseconds))

base_mults = [r for r in resolution_rows if r[0] == 28][0][1]
base_ms = [r for r in resolution_rows if r[0] == 28][0][2]
for side, mults, milliseconds in resolution_rows:
    print("%8s %12s %10.3f %11.2f× %9.2f×"
          % ("%d×%d" % (side, side),
             format(mults, ",").replace(",", " "), milliseconds,
             mults / base_mults, milliseconds / base_ms))

print()
biggest = resolution_rows[-1]
print("Від 28×28 до 56×56 множень стало в %.2f раза більше — сторона подвоїлась,"
      % (biggest[1] / base_mults))
print("площа зросла вчетверо. А часу — лише в %.2f раза." % (biggest[2] / base_ms))
print("Кількість множень — міра роботи, а не швидкості.")

Тепер із навчанням: та сама мережа на вході 20×20 замість 28×28.

⏱ Ще **одна** мережа — близько десяти секунд.

In [ ]:
small_input_run = run_experiment("вхід 20×20", width=1.0, separable=False, side=20)
experiments.append(small_input_run)

print()
print("%-14s %10s %12s %10s %10s"
      % ("вхід", "параметрів", "множень", "прохід,мс", "точність"))
print("-" * 62)
for run in (regular_run, small_input_run):
    print("%-14s %10s %12s %10.2f %10.3f"
          % ("%d×%d" % (run["side"], run["side"]),
             format(run["params"], ",").replace(",", " "),
             format(run["mults"], ",").replace(",", " "),
             run["ms"], run["accuracy"]))
print()
print("Параметрів рівно стільки ж — і це не збіг, а наслідок глобального усереднення.")
print("Множень удвічі менше, часу менше в %.2f раза, точність нижча на %.3f."
      % (regular_run["ms"] / small_input_run["ms"],
         regular_run["accuracy"] - small_input_run["accuracy"]))

## 8 · Що з цього обирати

Останній крок — повернутися до девʼяти справжніх архітектур і подивитись на них
уже не як на список, а як на кандидатури під конкретний бюджет. Правило просте:
**спершу відсікаємо тих, хто не вкладається, і лише потім обираємо серед решти**.

In [ ]:
SCENARIOS = [
    ("сервер, пакетна обробка", 600, 1000),
    ("сервер, відповідь на запит", 300, 120),
    ("телефон, кадр із камери", 30, 35),
    ("браузер", 15, 25),
]

for title, mb_limit, ms_limit in SCENARIOS:
    passed = [row["name"] for row in measured
              if row["mb"] <= mb_limit and row["ms"] <= ms_limit]
    print("%-28s ≤ %3d МБ, ≤ %3d мс  →  %d з 9" % (title, mb_limit, ms_limit, len(passed)))
    print("%-28s %s" % ("", ", ".join(passed) if passed else "не проходить ніхто"))
    print()

print("Зверни увагу: при жорсткому бюджеті часу відсіюються не найважчі мережі,")
print("а найглибші. Inception-v3 (%.0f МБ) вилітає раніше за AlexNet (%.0f МБ)."
      % ([r for r in measured if r["name"] == "Inception-v3"][0]["mb"],
         [r for r in measured if r["name"] == "AlexNet"][0]["mb"]))

In [ ]:
print("усього навчено мереж:", len(experiments))
print("сумарний час навчання: %.1f с" % sum(r["train_s"] for r in experiments))
print("увесь зошит виконався за %.0f с" % (time.perf_counter() - notebook_started))

## Що робити далі

### 🟢 Рівень 1
Додай у таблицю розділу 1 ще дві архітектури — наприклад `models.resnet34`
і `models.efficientnet_b1` — і подивись, чи ламають вони порядок за часом.

### 🟡 Рівень 2
Постав у `ShapeNet` ядро 5×5 замість 3×3 і повтори замір розділу 5. Економія
роздільної згортки має вирости приблизно до 25 разів за параметрами. А за часом?

### 🔴 Рівень 3
Побудуй мережу під бюджет **часу**, а не параметрів: не більше 4 мілісекунд на
батч із 64 зображень. Крути обидві ручки — ширину й роздільність — і покажи
заміром, що бюджет дотримано.

Повне домашнє завдання — у [homework.html](homework.html).